# 1. Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# 2. Read Bronze Table

In [0]:
df = spark.table('workspace.bronze.crm_prd_info')

# 3. Silver Layer Transformations 

## 3.1. Trim Whitespace from All String Columns in Dataframe

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## 3.2. Product Key Parsing

In [0]:
# Create new column `cat_id` from `prd_key`
df = df.withColumn('cat_id', F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))

# Remove prefix (ie. category id) from `prd_key
df = df.withColumn('prd_key', F.substring(col('prd_key'), 7, F.length(col('prd_key'))))

## 3.3. Cost Cleanup

In [0]:
df = df.withColumn('prd_cost', F.coalesce(col('prd_cost'), F.lit(0)))

## 3.4. Product Line Normalization

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

## 3.5. Date Casting

In [0]:
df = df.withColumn('prd_start_dt', col('prd_start_dt').cast(DateType()))
df = df.withColumn('prd_end_dt', col('prd_end_dt').cast(DateType()))

## 3.6. Handle Product End Date 

In [0]:
# Define window specification
window_spec = Window.partitionBy("prd_key").orderBy("prd_start_dt")

# Apply transformation to dataframe
df = df.withColumn(
    "prd_end_dt",
    F.date_sub(F.lead("prd_start_dt").over(window_spec), 1)
)

## 3.7. Rename DataFrame Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
# Sanity Checks of Dataframe
df.limit(10).display()

# 4. Writing Table To Silver Layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

In [0]:
%sql
-- Sanity Checks of Silver Table 
SELECT * FROM silver.crm_products LIMIT 10